# 08 — Évaluation finale
Agrège le dernier run d'ablation. Aucun chiffre n'est produit sans run réel.

In [ ]:
# --- Bootstrap (Kaggle or local) -------------------------------------------------
# Kaggle: Settings > Internet = ON ; Add-ons > Secrets > add "MODEL_API_KEY" (and "GITHUB_TOKEN" if the repo is private)
import os, subprocess, sys
REPO = "code-htech07/sina-meta"
BRANCH = "ccr-c8fab4b3-0jmvi1"  # change to "main" once merged
if os.path.exists("engine"):            # already inside the repo (local run)
    pass
else:
    if not os.path.exists("sina-meta"):
        url = f"https://github.com/{REPO}.git"
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = f"https://{tok}@github.com/{REPO}.git"
        except Exception:
            pass
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "sina-meta"], check=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # silent: never print a tokenised URL
    os.chdir("sina-meta")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())
from engine.config import describe_secret_status
from engine.logging_utils import setup_logging
setup_logging("WARNING")
print(describe_secret_status())  # shows only presence/length of the key, never its value

In [ ]:
import json, pandas as pd
from pathlib import Path
runs = sorted(Path('experiments/ablation/runs').glob('*/summary.json'))
assert runs, 'Aucun run : exécuter le notebook 07'
summary = json.loads(runs[-1].read_text())
rows = {k: {**v['classification'], **{'cssvd_top1': v['cssvd_specific'].get('top1_hit_rate')}, **v['quality'], **v['cost_latency']} for k, v in summary['experiments'].items()}
pd.DataFrame(rows).T

In [ ]:
print(open(runs[-1].parent / 'REPORT.md').read())